# DeepONet


In [1]:
# ======================================================================
# DeepONet Demo: Learning Implied Volatility Surfaces (Heston-like model)
# Author: ChatGPT (2025)
# ======================================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import trange
from scipy.stats import norm

# ============================================================
# 1.  Black–Scholes Helper Functions
# ============================================================

def bs_price(S0, K, T, sigma, r=0.0):
    """Standard Black–Scholes call price."""
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def implied_vol_call(price, S0, K, T, r=0.0):
    """Simple Newton solver for implied vol (not used for training)."""
    sigma = 0.2
    for _ in range(50):
        price_est = bs_price(S0, K, T, sigma, r)
        vega = S0 * norm.pdf((np.log(S0/K)+(r+0.5*sigma**2)*T)/(sigma*np.sqrt(T))) * np.sqrt(T)
        sigma -= (price_est - price) / max(vega, 1e-8)
        sigma = np.clip(sigma, 1e-4, 5.0)
    return sigma

# ============================================================
# 2.  Simplified "Heston-like" IV Generator
# ============================================================

def heston_iv_surface(k_grid, t_grid, kappa, theta, sigma, rho, v0):
    """
    Analytic-inspired toy function that produces structured IV surfaces.
    Uses a mean-reverting base vol + smile + term structure.
    """
    K, T = np.meshgrid(k_grid, t_grid)
    moneyness = K - 1.0  # linear moneyness, centered at ATM (K=1)
    base = np.sqrt(theta * (1 - np.exp(-kappa * T)) + v0 * np.exp(-kappa * T))
    smile = 1 + rho * moneyness + 0.05 * np.sin(np.pi * T)
    return np.abs(base * smile * (1 + 0.5 * sigma * moneyness**2))

# ============================================================
# 3.  Generate Synthetic Dataset with Randomized Grids
# ============================================================

n_surfaces = 20000
base_k_grid = np.linspace(0.5, 1.5, 9)
base_t_grid = np.linspace(0.05, 1.0, 9)
S0 = 1.0

def sample_params():
    kappa = np.random.uniform(0.5, 3.0)
    theta = np.random.uniform(0.02, 0.15)
    sigma = np.random.uniform(0.2, 0.8)
    rho   = np.random.uniform(-0.8, 0.0)
    v0    = np.random.uniform(0.02, 0.15)
    return np.array([kappa, theta, sigma, rho, v0])

X_branch, X_trunk, Y = [], [], []

for _ in trange(n_surfaces, desc="Generating surfaces"):
    params = sample_params()

    # --- Randomized K grid (±½ spacing jitter around base) ---
    ΔK = base_k_grid[1] - base_k_grid[0]
    k_grid_i = base_k_grid + np.random.uniform(-0.5 * ΔK, 0.5 * ΔK, size=len(base_k_grid))
    k_grid_i = np.clip(k_grid_i, 0.5, 1.5)

    # --- Randomized T grid (±½ spacing jitter around base) ---
    ΔT = base_t_grid[1] - base_t_grid[0]
    t_grid_i = base_t_grid + np.random.uniform(-0.5 * ΔT, 0.5 * ΔT, size=len(base_t_grid))
    t_grid_i = np.clip(t_grid_i, 0.01, None)

    # --- Generate surface ---
    iv = heston_iv_surface(k_grid_i, t_grid_i, *params).flatten()
    K, T = np.meshgrid(k_grid_i, t_grid_i)
    coords = np.stack([K.flatten(), T.flatten()], axis=1)

    X_branch.append(np.repeat(params[None, :], len(coords), axis=0))
    X_trunk.append(coords)
    Y.append(iv[:, None])

X_branch = np.concatenate(X_branch, axis=0)
X_trunk  = np.concatenate(X_trunk, axis=0)
Y        = np.concatenate(Y, axis=0)

# ============================================================
# 3.5  Surface-wise Train/Test Split
# ============================================================

points_per_surface = len(base_k_grid) * len(base_t_grid)
surface_indices = np.arange(n_surfaces)
np.random.shuffle(surface_indices)

n_train_surfaces = int(0.8 * n_surfaces)
train_surfaces = surface_indices[:n_train_surfaces]
test_surfaces  = surface_indices[n_train_surfaces:]

train_idx = np.concatenate([
    np.arange(i * points_per_surface, (i + 1) * points_per_surface)
    for i in train_surfaces
])
test_idx = np.concatenate([
    np.arange(i * points_per_surface, (i + 1) * points_per_surface)
    for i in test_surfaces
])

Xb_train = torch.tensor(X_branch[train_idx], dtype=torch.float32)
Xt_train = torch.tensor(X_trunk[train_idx], dtype=torch.float32)
y_train  = torch.tensor(Y[train_idx], dtype=torch.float32)

Xb_test = torch.tensor(X_branch[test_idx], dtype=torch.float32)
Xt_test = torch.tensor(X_trunk[test_idx], dtype=torch.float32)
y_test  = torch.tensor(Y[test_idx], dtype=torch.float32)

# ============================================================
# 4.  Define DeepONet Architecture
# ============================================================

latent_dim = 64

class BranchNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, output_dim)
        )
    def forward(self, x):
        return self.net(x)

class TrunkNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, output_dim)
        )
    def forward(self, x):
        return self.net(x)

branch = BranchNet(5, latent_dim)
trunk  = TrunkNet(2, latent_dim)

def deeponet_forward(branch, trunk, xb, xt):
    b_out = branch(xb)
    t_out = trunk(xt)
    return torch.sum(b_out * t_out, dim=1, keepdim=True)

# ============================================================
# 5.  Training
# ============================================================

optimizer = optim.Adam(list(branch.parameters()) + list(trunk.parameters()), lr=1e-3)
loss_fn = nn.MSELoss()

epochs = 10
batch_size = 1024
n_train = len(train_idx)

for epoch in range(epochs):
    perm = torch.randperm(n_train)
    total_loss = 0.0

    for i in range(0, n_train, batch_size):
        idx = perm[i:i + batch_size]
        xb, xt, y = Xb_train[idx], Xt_train[idx], y_train[idx]

        y_pred = deeponet_forward(branch, trunk, xb, xt)
        loss = loss_fn(y_pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(idx)

    with torch.no_grad():
        y_val = deeponet_forward(branch, trunk, Xb_test, Xt_test)
        val_loss = loss_fn(y_val, y_test).item()

    print(f"Epoch {epoch+1:02d}: train_loss={total_loss/n_train:.6f}, val_loss={val_loss:.6f}")




Generating surfaces: 100%|██████████| 20000/20000 [00:01<00:00, 10944.03it/s]


Epoch 01: train_loss=0.000898, val_loss=0.000091
Epoch 02: train_loss=0.000056, val_loss=0.000037


KeyboardInterrupt: 

In [3]:
# ============================================================
# 6.  Visualization (classic layout, aligned colorbars)
# ============================================================

# Select one random test surface (each surface = 81 points)
idx_example = np.random.randint(0, len(Xb_test)//81)
idxs = slice(idx_example*81, (idx_example+1)*81)

# Extract parameters used for that surface
params_example = Xb_test[idxs][0].numpy()
param_names = ["kappa", "theta", "sigma", "rho", "v0"]

# Predict implied vol surface for this parameter set
with torch.no_grad():
    pred = deeponet_forward(branch, trunk, Xb_test[idxs], Xt_test[idxs]).numpy()
true_iv = y_test[idxs].numpy()

# Reshape to 9x9 grid for plotting
true_iv_2d = true_iv.reshape(9,9)
pred_iv_2d = pred.reshape(9,9)
diff_2d = pred_iv_2d - true_iv_2d

# Shared limits for True & Pred
vmin = min(true_iv.min(), pred.min())
vmax = max(true_iv.max(), pred.max())
diff_lim = np.max(np.abs(diff_2d))

# # --- Create figure ---
# fig, ax = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={'wspace': 0.25})

# # --- (1) True IV Surface ---
# im0 = ax[0].imshow(true_iv_2d, origin='lower', extent=[0.5, 1.5, 0.05, 1.0],
#                    aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
# ax[0].set_title("True IV Surface")
# ax[0].set_xlabel("Strike")
# ax[0].set_ylabel("Maturity (T)")
# fig.colorbar(im0, ax=ax[0], label="Implied Volatility")

# # --- (2) Predicted IV Surface ---
# im1 = ax[1].imshow(pred_iv_2d, origin='lower', extent=[0.5, 1.5, 0.05, 1.0],
#                    aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
# ax[1].set_title("Predicted IV Surface (DeepONet)")
# ax[1].set_xlabel("Strike")
# ax[1].set_ylabel("Maturity (T)")
# fig.colorbar(im1, ax=ax[1], label="Implied Volatility")

# # --- (3) Difference ---
# im2 = ax[2].imshow(diff_2d, origin='lower', extent=[0.5, 1.5, 0.05, 1.0],
#                    aspect='auto', cmap='coolwarm', vmin=-diff_lim, vmax=diff_lim)
# ax[2].set_title("Difference (Pred - True)")
# ax[2].set_xlabel("Strike")
# ax[2].set_ylabel("Maturity (T)")
# fig.colorbar(im2, ax=ax[2], label="Error in Volatility")

# # --- Global title ---
# plt.suptitle(
#     f"Example Random Test Surface (#{idx_example})\n"
#     + ", ".join([f"{n}={p:.3f}" for n,p in zip(param_names, params_example)]),
#     fontsize=11, y=1.05
# )

# plt.tight_layout()
# plt.show()


In [ ]:
Xt_test[idxs]

In [4]:
# ============================================================
# 7. Demonstration: TrunkNet with variable (K,T) inputs
# ============================================================

def predict_surface(branch, trunk, params, K_points, T_points):
    """
    Predict implied volatilities for arbitrary (K,T) inputs
    using a single parameter vector (Branch input).
    """
    # prepare inputs
    θ = torch.tensor(params[None, :], dtype=torch.float32)
    b_out = branch(θ).detach()  # [1, latent_dim]

    # build (K,T) coordinate tensor of arbitrary shape
    KT = np.stack([K_points, T_points], axis=1)
    KT_tensor = torch.tensor(KT, dtype=torch.float32)

    # predict via trunk
    t_out = trunk(KT_tensor)  # [N, latent_dim]
    σ_pred = (t_out @ b_out.T).squeeze().detach().numpy()  # [N]
    return KT, σ_pred


# --- Example 1: Regular 9x9 grid (baseline) ---
K_grid = np.linspace(0.8, 1.2, 9)
T_grid = np.linspace(0.05, 1.0, 9)
K_mesh, T_mesh = np.meshgrid(K_grid, T_grid)
KT_regular, σ_regular = predict_surface(branch, trunk, params_example,
                                        K_mesh.flatten(), T_mesh.flatten())

# --- Example 2: Only 10 random points ---
K_rand = np.random.uniform(0.85, 1.15, 10)
T_rand = np.random.uniform(0.05, 1.0, 10)
KT_random, σ_random = predict_surface(branch, trunk, params_example, K_rand, T_rand)

# --- Example 3: Fine 25x25 grid ---
K_fine = np.linspace(0.8, 1.2, 25)
T_fine = np.linspace(0.05, 1.0, 25)
K_mesh_fine, T_mesh_fine = np.meshgrid(K_fine, T_fine)
KT_fine, σ_fine = predict_surface(branch, trunk, params_example,
                                  K_mesh_fine.flatten(), T_mesh_fine.flatten())


# --- Visualization ---
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# Regular grid
im0 = ax[0].imshow(σ_regular.reshape(9,9), origin='lower',
                   extent=[0.5, 1.5, 0.05, 1.0], aspect='auto', cmap='viridis')
ax[0].set_title("Regular 9×9 Grid")

# Random points (scatter)
sc = ax[1].scatter(KT_random[:,0], KT_random[:,1], c=σ_random, cmap='viridis', s=80)
ax[1].set_title("Random (K,T) Points")
fig.colorbar(sc, ax=ax[1], label="Implied Volatility")

# Fine grid
im2 = ax[2].imshow(σ_fine.reshape(25,25), origin='lower',
                   extent=[0.5, 1.5, 0.05, 1.0], aspect='auto', cmap='viridis')
ax[2].set_title("Fine 25×25 Grid")

for a in ax:
    a.set_xlabel("Strike"); a.set_ylabel("Maturity (T)")

fig.colorbar(im0, ax=ax[0], label="Implied Volatility")
fig.colorbar(im2, ax=ax[2], label="Implied Volatility")

plt.suptitle("DeepONet Inference on Variable (K,T) Grids", fontsize=13, y=1.05)
plt.tight_layout()
plt.show()


: 

# FNO

In [23]:
# ======================================================================
# Fourier Neural Operator (FNO) for Heston-like Implied Volatility Surfaces
# Author: ChatGPT (2025)
# ======================================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import trange

# ============================================================
# 1. Synthetic Heston-like IV generator
# ============================================================

def heston_iv_surface(k_grid, t_grid, kappa, theta, sigma, rho, v0):
    """Toy analytic-style IV surface generator."""
    K, T = np.meshgrid(k_grid, t_grid)
    m = K - 1.0
    base = np.sqrt(theta * (1 - np.exp(-kappa * T)) + v0 * np.exp(-kappa * T))
    smile = 1 + rho * m + 0.05 * np.sin(np.pi * T)
    iv = np.abs(base * smile * (1 + 0.5 * sigma * m**2))
    return iv.astype(np.float32)  # shape [len(T), len(K)]

def sample_params():
    """Sample random Heston-like parameters."""
    kappa = np.random.uniform(0.5, 3.0)
    theta = np.random.uniform(0.02, 0.15)
    sigma = np.random.uniform(0.2, 0.8)
    rho   = np.random.uniform(-0.8, 0.0)
    v0    = np.random.uniform(0.02, 0.15)
    return np.array([kappa, theta, sigma, rho, v0], dtype=np.float32)

# ============================================================
# 2. Dataset
# ============================================================

class IVDataset(torch.utils.data.Dataset):
    def __init__(self, n_surfaces=20000, H=64, W=64):
        super().__init__()
        self.n_surfaces = n_surfaces
        self.H, self.W = H, W

        # Fixed uniform grids
        self.K_grid = np.linspace(0.5, 1.5, W, dtype=np.float32)
        self.T_grid = np.linspace(0.0, 1.0, H, dtype=np.float32)
        self.KU, self.TU = np.meshgrid(self.K_grid, self.T_grid)

        self.params = np.stack([sample_params() for _ in range(n_surfaces)], axis=0)

    def __len__(self):
        return self.n_surfaces

    def __getitem__(self, idx):
        p = self.params[idx]
        iv = heston_iv_surface(self.K_grid, self.T_grid, *p)  # [H, W]

        # broadcast parameters + coordinates as channels
        C_param = np.stack([np.full_like(iv, pi) for pi in p], axis=0)  # [5, H, W]
        C_K = self.KU[None, ...]
        C_T = self.TU[None, ...]
        x = np.concatenate([C_param, C_K, C_T], axis=0)  # [7, H, W]
        y = iv[None, ...]  # [1, H, W]
        return torch.from_numpy(x), torch.from_numpy(y)

# ============================================================
# 3. Fourier Layer + FNO2d
# ============================================================

class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = 0.02
        self.weight = nn.Parameter(self.scale * torch.randn(in_channels, out_channels, modes1, modes2, 2))

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfft2(x, norm="ortho")
        out_ft = torch.zeros(batchsize, self.out_channels, x.size(-2), x.size(-1)//2 + 1,
                             dtype=torch.complex64, device=x.device)

        w = self.weight
        wr, wi = w[..., 0], w[..., 1]
        w_complex = torch.complex(wr, wi)
        m1, m2 = min(self.modes1, x.size(-2)), min(self.modes2, x.size(-1)//2 + 1)
        out_ft[:, :, :m1, :m2] = torch.einsum("bixy,ioxy->boxy",
                                              x_ft[:, :, :m1, :m2],
                                              w_complex[:, :, :m1, :m2])
        x = torch.fft.irfft2(out_ft, s=(x.size(-2), x.size(-1)), norm="ortho")
        return x

class FNO2d(nn.Module):
    def __init__(self, in_channels=7, out_channels=1, width=64, modes1=8, modes2=8, depth=4):
        super().__init__()
        self.fc0 = nn.Conv2d(in_channels, width, 1)
        self.convs = nn.ModuleList([SpectralConv2d(width, width, modes1, modes2) for _ in range(depth)])
        self.ws = nn.ModuleList([nn.Conv2d(width, width, 1) for _ in range(depth)])
        self.fc1 = nn.Conv2d(width, width, 1)
        self.fc2 = nn.Conv2d(width, out_channels, 1)
        self.act = nn.GELU()

    def forward(self, x):
        x = self.fc0(x)
        for conv, w in zip(self.convs, self.ws):
            x = self.act(conv(x) + w(x))
        x = self.act(self.fc1(x))
        x = self.fc2(x)
        return x

# ============================================================
# 4. Training setup
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n_surfaces = 20000
dataset = IVDataset(n_surfaces=n_surfaces, H=9, W=9)
idx = np.arange(n_surfaces)
np.random.shuffle(idx)
n_train = int(0.8 * n_surfaces)
train_ds = torch.utils.data.Subset(dataset, idx[:n_train])
test_ds  = torch.utils.data.Subset(dataset, idx[n_train:])

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=32, shuffle=False)

model = FNO2d(in_channels=7, out_channels=1, width=64, modes1=8, modes2=8, depth=4).to(device)
optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

# ============================================================
# 5. Training loop
# ============================================================

epochs = 15
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = criterion(pred, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
    train_loss = total_loss / len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            val_loss += criterion(pred, y).item() * X.size(0)
    val_loss /= len(test_loader.dataset)
    print(f"Epoch {epoch:02d}: train_loss={train_loss:.6f}, val_loss={val_loss:.6f}")

# ============================================================
# 6. Evaluation
# ============================================================

model.eval()
with torch.no_grad():
    X, y = next(iter(test_loader))
    X, y = X.to(device), y.to(device)
    pred = model(X)
    print("Pred mean/std:", pred.mean().item(), pred.std().item())
    print("True mean/std:", y.mean().item(), y.std().item())


Epoch 01: train_loss=0.001628, val_loss=0.000102
Epoch 02: train_loss=0.000029, val_loss=0.000005
Epoch 03: train_loss=0.000017, val_loss=0.000061
Epoch 04: train_loss=0.000010, val_loss=0.000004
Epoch 05: train_loss=0.000014, val_loss=0.000029
Epoch 06: train_loss=0.000011, val_loss=0.000010
Epoch 07: train_loss=0.000020, val_loss=0.000062
Epoch 08: train_loss=0.000005, val_loss=0.000005
Epoch 09: train_loss=0.000010, val_loss=0.000025
Epoch 10: train_loss=0.000009, val_loss=0.000015
Epoch 11: train_loss=0.000013, val_loss=0.000001
Epoch 12: train_loss=0.000014, val_loss=0.000002
Epoch 13: train_loss=0.000002, val_loss=0.000000
Epoch 14: train_loss=0.000010, val_loss=0.000003
Epoch 15: train_loss=0.000016, val_loss=0.000040
Pred mean/std: 0.30849215388298035 0.06469669938087463
True mean/std: 0.3081306219100952 0.06493711471557617
